# WriteWise — Stage 1: MobileNetV2 Fine-Tuning on CCC

Fine-tunes MobileNetV2 (ImageNet-pretrained) on the CCC cursive character dataset
to learn cursive-specific stroke features. See `ML_PIPELINE.md §4` for full details.

**This model is never deployed as-is** — only its convolutional backbone carries
forward into Stage 2's regression head.

## Prerequisites
- Processed CCC dataset (from `convert_ccc.py`) uploaded to Google Drive
- Colab runtime set to **GPU** (Runtime → Change runtime type → T4 GPU)


## 1. Setup


In [ ]:
# Mount Google Drive for data and checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

# Configuration — update these paths to match your Drive layout
DATA_DIR = '/content/drive/MyDrive/writewise/training/data/processed'
CHECKPOINT_DIR = '/content/drive/MyDrive/writewise/training/checkpoints'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


In [ ]:
import numpy as np
import tensorflow as tf
from pathlib import Path

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')


## 2. Data Loading


In [ ]:
def load_split(split_dir: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Load .npy files from a split directory.
    
    Returns (images, labels_encoded, class_names).
    """
    split_path = Path(split_dir)
    images = []
    labels = []
    
    for npy_file in sorted(split_path.glob('*.npy')):
        img = np.load(npy_file)
        # Extract label from filename: <label>_<source>.npy
        label = npy_file.stem.split('_')[0]
        images.append(img)
        labels.append(label)
    
    images = np.array(images, dtype=np.uint8)
    
    # Encode labels to integers
    class_names = sorted(set(labels))
    label_to_idx = {name: idx for idx, name in enumerate(class_names)}
    labels_encoded = np.array([label_to_idx[l] for l in labels])
    
    return images, labels_encoded, class_names

print('Loading training data...')
X_train, y_train, class_names = load_split(f'{DATA_DIR}/train')
print(f'  Train: {X_train.shape}, {len(class_names)} classes')

print('Loading validation data...')
X_val, y_val, _ = load_split(f'{DATA_DIR}/val')
print(f'  Val: {X_val.shape}')

NUM_CLASSES = len(class_names)
print(f'\nClasses ({NUM_CLASSES}): {class_names}')


## 3. Preprocessing Pipeline


In [ ]:
INPUT_SIZE = 96  # ML_PIPELINE §2.3
BATCH_SIZE = 32

def preprocess_dataset(images: np.ndarray, labels: np.ndarray, augment: bool = False):
    """Build a tf.data.Dataset with preprocessing and optional augmentation.
    
    Preprocessing: grayscale → 3-channel → normalize [-1, 1].
    Augmentation (ML_PIPELINE §4.2): rotation ±15°, slight zoom/translate.
    """
    # Grayscale → 3-channel (MobileNetV2 expects RGB)
    images_3ch = np.stack([images] * 3, axis=-1)
    
    # Normalize to [-1, 1] (MobileNetV2 convention)
    images_norm = (images_3ch.astype(np.float32) / 127.5) - 1.0
    
    # One-hot encode labels
    labels_onehot = tf.keras.utils.to_categorical(labels, NUM_CLASSES)
    
    ds = tf.data.Dataset.from_tensor_slices((images_norm, labels_onehot))
    
    if augment:
        # ML_PIPELINE §4.2: rotation capped at ±15°, slight scale/translate
        augmentation = tf.keras.Sequential([
            tf.keras.layers.RandomRotation(
                factor=15/360,  # ±15 degrees
                fill_mode='constant',
                fill_value=-1.0,  # white in [-1,1] space
            ),
            tf.keras.layers.RandomZoom(
                height_factor=(-0.1, 0.1),
                width_factor=(-0.1, 0.1),
                fill_mode='constant',
                fill_value=-1.0,
            ),
            tf.keras.layers.RandomTranslation(
                height_factor=0.05,
                width_factor=0.05,
                fill_mode='constant',
                fill_value=-1.0,
            ),
        ])
        ds = ds.map(
            lambda x, y: (augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
        ds = ds.shuffle(buffer_size=1000)
    
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = preprocess_dataset(X_train, y_train, augment=True)
val_ds = preprocess_dataset(X_val, y_val, augment=False)

print('Datasets built.')
for batch_x, batch_y in train_ds.take(1):
    print(f'  Batch shape: {batch_x.shape}, Labels shape: {batch_y.shape}')


## 4. Model Architecture


In [ ]:
# MobileNetV2 backbone (ML_PIPELINE §3)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
    include_top=False,
    weights='imagenet',
)

# Classification head
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

print(f'Base model layers: {len(base_model.layers)}')
print(f'Total model parameters: {model.count_params():,}')


## 5. Phase A — Head-Only Training (ML_PIPELINE §4.1)

Freeze the entire MobileNetV2 base. Train only the classification head
for a few epochs to let it adapt without disturbing pretrained weights.


In [ ]:
# Freeze the entire base
base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print('Phase A: Head-only training')
print(f'  Trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}')

history_a = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

print(f'\nPhase A complete.')
print(f'  Final train accuracy: {history_a.history["accuracy"][-1]:.4f}')
print(f'  Final val accuracy:   {history_a.history["val_accuracy"][-1]:.4f}')


## 6. Phase B — Partial Unfreeze (ML_PIPELINE §4.1)

Unfreeze the top ~30% of MobileNetV2 layers and continue training at a
much lower learning rate. Early stopping on validation loss.


In [ ]:
# Unfreeze top ~30% of the base model
base_model.trainable = True
total_layers = len(base_model.layers)
freeze_until = int(total_layers * 0.7)

for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfroze {trainable_count}/{total_layers} base layers')

# Recompile with lower learning rate (ML_PIPELINE §4.3)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'Total trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}')

# Callbacks
checkpoint_path = f'{CHECKPOINT_DIR}/stage1_best.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

print('\nPhase B: Fine-tuning with early stopping')
history_b = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
)

print(f'\nPhase B complete.')
print(f'  Best val loss: {min(history_b.history["val_loss"]):.4f}')
print(f'  Best val accuracy: {max(history_b.history["val_accuracy"]):.4f}')
print(f'  Checkpoint saved to: {checkpoint_path}')


## 7. Training Summary


In [ ]:
import matplotlib.pyplot as plt

# Combine histories
all_acc = history_a.history['accuracy'] + history_b.history['accuracy']
all_val_acc = history_a.history['val_accuracy'] + history_b.history['val_accuracy']
all_loss = history_a.history['loss'] + history_b.history['loss']
all_val_loss = history_a.history['val_loss'] + history_b.history['val_loss']
phase_a_epochs = len(history_a.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(all_acc, label='Train')
ax1.plot(all_val_acc, label='Validation')
ax1.axvline(x=phase_a_epochs - 0.5, color='gray', linestyle='--', alpha=0.5, label='Phase A→B')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(all_loss, label='Train')
ax2.plot(all_val_loss, label='Validation')
ax2.axvline(x=phase_a_epochs - 0.5, color='gray', linestyle='--', alpha=0.5, label='Phase A→B')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/training_curves.png', dpi=150)
plt.show()

print(f'\nFinal results:')
print(f'  Best validation accuracy: {max(all_val_acc):.4f}')
print(f'  Best validation loss: {min(all_val_loss):.4f}')
print(f'  Checkpoint: {checkpoint_path}')
